# Backtester Prototype — run on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rohit-rakhecha/Backtester_Prototype/blob/PR/examples/run_in_jupyter.ipynb)

Click the badge above to open this exact notebook directly in Colab (it loads from GitHub —
no manual upload needed).

**Step through the cells one at a time (Shift+Enter)** rather than Runtime → Run all: section 3 below branches into two alternative ways to run the pipeline (3A interactive, upload your own paper; 3B fixed demo, no prompts) and you should run only one of them, so a blind "Run all" would execute both back to back. 3A also pauses at each prompt waiting for your input (upload a PDF, confirm Strategy Card fields, pick data sources, approve gates), so it isn't a fire-and-forget run regardless.

This notebook clones `rohit-rakhecha/Backtester_Prototype` (branch `PR`), installs dependencies, then lets you run the 8-stage AI Research Operating System pipeline either **on your own uploaded research paper** (3A) or against the attached, already-worked NSE factor-index example (3B). See `docs/PIPELINE.md` and `docs/DATA_SOURCES.md` for the full write-up of what each stage does and why.

Also works in a local Jupyter install — start Jupyter, open this file, and run the same cells.

## 1. Clone the repo

Colab's "Open in Colab" only loads this notebook file itself, not the rest of the repo --
so this cell fetches the actual code and data. **Safe and correct to re-run at any time**:
if the repo is already present (e.g. from an earlier run in this same Colab session), it
does a `git pull` instead of skipping, so you always get the latest fix/commit from
GitHub rather than a stale local copy.

In [ ]:
import os

REPO_URL = "https://github.com/rohit-rakhecha/Backtester_Prototype.git"
REPO_DIR = "/content/Backtester_Prototype" if os.path.isdir("/content") else "Backtester_Prototype"
BRANCH = "PR"

if os.path.basename(os.getcwd()) == os.path.basename(REPO_DIR) and os.path.isdir(".git"):
    print(f"Already inside {os.getcwd()} -- pulling latest {BRANCH}.")
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
elif os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already exists -- pulling latest {BRANCH} instead of re-cloning.")
    %cd {REPO_DIR}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

!git log -1 --oneline
!git status

# Force a fresh import of this package on the NEXT %run/import, even though
# the Colab kernel process itself is not restarted. Without this, a git pull
# updates the files on disk but Python keeps using the module objects already
# cached in sys.modules from an earlier run in this same session -- so re-running
# after a fix lands upstream would otherwise silently execute the OLD in-memory
# code and reproduce the exact same bug/traceback.
import sys
for _mod in list(sys.modules):
    if _mod == "backtester" or _mod.startswith("backtester."):
        del sys.modules[_mod]
print("Cleared cached backtester.* modules -- next run will use the freshly pulled code.")

## 2. Install dependencies

Colab already ships pandas/numpy/scipy/matplotlib, so this is fast. `cvxpy` and `pydantic`
are the ones Colab doesn't have by default.

In [ ]:
%pip install -q -r requirements.txt
%pip install -q pypdf pytest

## 3. Run the pipeline

Two ways to run this, both going through the exact same engine/validation code:

- **3A -- Interactive (recommended): upload your own paper.** Prompts you to upload a PDF, builds the Strategy Card interactively from what Stage 01 mines out of it, asks which India data sources to use (or upload) for whichever asset classes the paper touches on (equities, bonds, mutual funds, commodities, rates -- not just equities), then runs Stages 05-08. Use this to try a **different research paper**.
- **3B -- Fixed demo (fallback/regression check).** The original non-interactive worked example: a pre-filled Card + the attached NSE factor-index CSV. No prompts. Useful to confirm the engine itself still works if 3A behaves unexpectedly.

Run only ONE of 3A / 3B -- whichever you run leaves the same variable names (`card`, `results`, `dataset`, `library`, `audit`, `returns`, ...) in scope for the cells below, so sections 4-7 work identically either way.

### 3A. Interactive -- upload your own paper

Running this cell will pause repeatedly and ask you questions (upload a PDF, confirm/edit Strategy Card fields, pick or upload data sources, approve gates, ...). In Colab each prompt renders as an inline text box under the cell -- type your answer and press Enter (or just press Enter to accept the suggested default, shown in `[brackets]`, where there is one). File-upload prompts open Colab's native upload widget.

**Have your PDF ready before running this cell.**

In [ ]:
%run examples/run_pipeline_interactive.py

### 3B. Fixed demo (fallback -- no prompts)

The original worked example: a pre-filled Strategy Card replicating the attached "Simple Dynamic Stock/Bond/Gold Portfolios" paper's methodology, run against the attached NSE factor-index CSV. Only run this if you want the non-interactive baseline instead of 3A.

In [ ]:
%run examples/run_pipeline.py

## 4. Plot cumulative performance (the part a plain script can't give you)

Uses the `results` dict and `bench_value` series that `run_pipeline.py` left in scope.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 6))
for name, res in results.items():
    ax.plot(res.value.index, res.value.values, label=name, linewidth=1.5)
if "bench_value" in globals():  # only defined by 3B's fixed demo
    ax.plot(bench_value.index, bench_value.values, label="Benchmark (passive)", linewidth=1.5, linestyle="--", color="black")
ax.set_yscale("log")
ax.set_title("Cumulative portfolio value (log scale), net of India transaction costs")
ax.set_ylabel("Value (start = 1.0)")
ax.legend(loc="upper left", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Inspect the audit trail and promotion ladder directly

In [ ]:
import pandas as pd

audit_entries = audit.read_all()
pd.DataFrame(audit_entries)[["gate", "card_id", "reviewer", "decision", "note", "timestamp"]]

In [ ]:
pd.DataFrame(library.history(card.card_id))

## 6. Run the test suite (optional sanity check)

In [ ]:
!python -m pytest tests/ -v

## 7. Try a different volatility target or strategy (interactive exploration)

Everything below reuses the SAME deterministic engine (`backtester.engine.portfolio`)
and cost model that Stage 05 used above -- only the signal function changes, which is
exactly the "AI only authors the signal" boundary the pipeline enforces.

In [ ]:
from backtester.engine import signals as sig
from backtester.engine.portfolio import PortfolioSimulator, performance_metrics
from backtester.engine.costs import DEFAULT_INDIA_COST_MODEL
import numpy as np

for tv in [0.08, 0.12, 0.16, 0.20]:
    fn = sig.markowitz(equal_weight, tv, l1_trust_region=1.0, cov_lookback_days=11,
                        one_way_cost_bps=one_way_bps, cash_annual_rate=0.0)
    sim = PortfolioSimulator(asset_returns=returns, cost_model=DEFAULT_INDIA_COST_MODEL,
                              cash_annual_rate=0.0, rebalance_frequency="ME")
    r = sim.run(fn)
    m = performance_metrics(r.value)
    print(f"target_vol={tv:>5.0%}  return={m['return']:.2%}  vol={m['volatility']:.2%}  "
          f"sharpe={m['sharpe']:.2f}  maxDD={m['max_drawdown']:.2%}")